# MCSL-069 — Question 25

**Section:** 8.2 Machine Learning, Session 5, Exercise b

**Question:** Implement FP-tree growth for frequent-itemset mining.

The accompanying CSV file is a small, reproducible input dataset. Run the code cell below to see the implementation and its results.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from collections import Counter
import math
transactions = [row.split("|") for row in pd.read_csv("b_transactions.csv")["items"]]
minimum_support = 0.25
min_count = math.ceil(minimum_support * len(transactions))

class Node:
    def __init__(self, item, count=0, parent=None):
        self.item, self.count, self.parent = item, count, parent
        self.children = {}

def build_tree(weighted_paths):
    frequencies = Counter()
    for path, weight in weighted_paths:
        frequencies.update({item: weight for item in set(path)})
    frequent = {item: count for item, count in frequencies.items() if count >= min_count}
    if not frequent:
        return None, {}
    root = Node(None)
    header = {item: [] for item in frequent}
    for path, weight in weighted_paths:
        ordered = sorted((item for item in set(path) if item in frequent),
                         key=lambda item: (-frequent[item], item))
        node = root
        for item in ordered:
            if item not in node.children:
                node.children[item] = Node(item, parent=node)
                header[item].append(node.children[item])
            node = node.children[item]
            node.count += weight
    return root, header

patterns = {}
def fp_growth(weighted_paths, suffix=()):
    _, header = build_tree(weighted_paths)
    for item in sorted(header, key=lambda value: (sum(node.count for node in header[value]), value)):
        support = sum(node.count for node in header[item])
        pattern = tuple(sorted(suffix + (item,)))
        patterns[pattern] = support
        conditional_paths = []
        for node in header[item]:
            path = []
            parent = node.parent
            while parent is not None and parent.item is not None:
                path.append(parent.item)
                parent = parent.parent
            if path:
                conditional_paths.append((path, node.count))
        if conditional_paths:
            fp_growth(conditional_paths, suffix + (item,))

fp_growth([(transaction, 1) for transaction in transactions])
print("FP-tree frequent itemsets (support count):")
for items, count in sorted(patterns.items(), key=lambda pair: (len(pair[0]), pair[0])):
    print(items, count, f"({count / len(transactions):.2f})")


FP-tree frequent itemsets (support count):
('bread',) 10 (0.50)
('butter',) 7 (0.35)
('eggs',) 6 (0.30)
('milk',) 11 (0.55)
('tea',) 5 (0.25)
('bread', 'milk') 7 (0.35)
('butter', 'milk') 5 (0.25)
('eggs', 'milk') 5 (0.25)
